## Graph RAG with Neo4j and Azure OpenAI


<img src="./graph_rag_architecture.png">

### installing dependencies


In [1]:
2

2

In [2]:
%pip install requests python-dotenv neo4j

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import requests
from dotenv import load_dotenv
load_dotenv()

True

### LLM client (no OpenAI needed)
Set `LLM_PROVIDER` in `.env`: `ollama` (default, local/free), `anthropic`, or `openai` (any OpenAI-compatible API such as Groq).

In [4]:
import re

import time

def _post(url, retries=6, **kwargs):
    """requests.post with backoff on 429/5xx (honours Retry-After). Groq's free tier rate-limits aggressively."""
    for attempt in range(retries + 1):
        r = requests.post(url, **kwargs)
        if r.status_code != 429 and r.status_code < 500:
            return r
        if attempt == retries:
            return r
        wait = float(r.headers.get("retry-after", 0) or 0) or min(2 ** attempt * 2, 60)
        if wait > 120:  # daily quota exhausted: waiting won't help
            print(f"[{r.status_code}] asked to wait {wait:.0f}s - quota likely exhausted. {r.text[:200]}")
            return r
        print(f"[{r.status_code}] rate limited/server error, retrying in {wait:.0f}s ({attempt + 1}/{retries})")
        time.sleep(wait + 1)

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").lower()

def chat(system_prompt, user_prompt, temperature=0):
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}]
    if LLM_PROVIDER == "ollama":
        r = _post(os.getenv("OLLAMA_URL", "http://localhost:11434") + "/api/chat",
                          json={"model": os.getenv("LLM_MODEL", "llama3.1"), "messages": messages,
                                "stream": False, "options": {"temperature": temperature}}, timeout=300)
        r.raise_for_status()
        return r.json()["message"]["content"]
    if LLM_PROVIDER == "anthropic":
        r = _post("https://api.anthropic.com/v1/messages",
                          headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"],
                                   "anthropic-version": "2023-06-01"},
                          json={"model": os.getenv("LLM_MODEL", "claude-haiku-5-5"), "max_tokens": 2000,
                                "system": system_prompt, "temperature": temperature,
                                "messages": messages[1:]}, timeout=120)
        r.raise_for_status()
        return r.json()["content"][0]["text"]
    if LLM_PROVIDER == "openai":  # OpenAI-compatible (Groq, GitHub Models, etc.)
        r = _post(os.environ["OPENAI_BASE_URL"].rstrip("/") + "/chat/completions",
                          headers={"Authorization": "Bearer " + os.environ["OPENAI_API_KEY"]},
                          json={"model": os.environ["LLM_MODEL"], "messages": messages,
                                "temperature": temperature}, timeout=120)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"]
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}")

def clean_cypher(text):
    """Strip markdown code fences LLMs like to add around Cypher."""
    return re.sub(r"^```(?:cypher)?|```$", "", text.strip(), flags=re.MULTILINE).strip()

In [5]:
ontology_list = []

### Creating a function to identify nodes and their relationships
the code will send a call to GPT engine and the GPT engine will output nodes and relationships based upon the ontology defined in the prompt we feed
it with

In [6]:
def identitfy_relationships_and_nodes(file_text):
    
    system_prompt = f"""Assistant is a Named Entity Recognition (NER) expert. The assistant can identify named entities 
    such as a person, place, or thing. The assistant can also identify entity relationships, which describe
    how entities relate to each other (eg: married to, located in, held by). Identify the named entities
    and the entity relationships present in the text by returning comma separated list of tuples
    representing the relationship between two entities in the format (entity, relationship, entity). Only
    generate tuples from the list of entities and the possible entity relationships listed below. Return
    only generated tuples in a comma separated tuple separated by a new line for each tuple.

    Entities:
    - Hotel
    - Location
    - Facilities
    - CustomerTyoe
    - Reviewer

    Relationships:
    - [Hotel],is_located_in,[Location]
    - [Hotel],has_facilities,[Facilities]
    - [Hotel],has_customers,[CustomerType]
    - [Hotel],has_reviewer,[Reviewer]

    Example Output:
    Creek Hotel,is_located_in,Dubai
    Creek Hotel,has_facilities,swimming pool
    Creek Hotel,has_customers,Businessmen
    Creek Hotel,has_customers,senior citizens
    Creek Hotel,has_reviewer,John Doe

    """

    user_prompt = f"""Identify the named entities and entity relationships in the hotel review text above. Return the
    entities and entity relationships in a tuple separated by commas. Return only generated tuples in a
    comma separated tuple separated by a new line for each tuple.

    Text: {file_text}"""


    
    content = chat(system_prompt, user_prompt)
    ontology_list.append(content)
    print(content)

In [7]:
with open("./reviews/201801.txt", "r") as file:
    file_text = file.read()
    identitfy_relationships_and_nodes(file_text)
    
with open("./reviews/201802.txt", "r") as file:
    file_text = file.read()
    identitfy_relationships_and_nodes(file_text)

with open("./reviews/201803.txt", "r") as file:
    file_text = file.read()
    identitfy_relationships_and_nodes(file_text)

with open("./reviews/201804.txt", "r") as file:
    file_text = file.read()
    identitfy_relationships_and_nodes(file_text)

with open("./reviews/201805.txt", "r") as file:
    file_text = file.read()
    identitfy_relationships_and_nodes(file_text)
    
    


Creek Hotel,is_located_in,Dubai
Creek Hotel,is_located_in,Bur Dubai
Creek Hotel,has_facilities,sofa
Creek Hotel,has_facilities,TV
Creek Hotel,has_facilities,hot-shower bath
Creek Hotel,has_customers,businessmen
Creek Hotel,has_customers,tourists
Creek Hotel,has_reviewer,Ryouta Sato
The Buckingham Hotel,is_located_in,New York City
The Buckingham Hotel,has_facilities,sofa
The Buckingham Hotel,has_facilities,TV
The Buckingham Hotel,has_facilities,hot-shower bath
The Buckingham Hotel,has_reviewer,Issac Collett
The Buckingham Hotel,is_located_in,London, UK
The Buckingham Hotel,has_facilities,rooms
The Buckingham Hotel,has_facilities,courtyard
The Buckingham Hotel,has_facilities,restaurant
The Buckingham Hotel,has_facilities,Indian cuisine
The Buckingham Hotel,has_facilities,Michelin Star
The Buckingham Hotel,has_facilities,kitchen
The Buckingham Hotel,has_facilities,lounge
The Buckingham Hotel,has_facilities,bedroom
The Buckingham Hotel,has_facilities,bathroom
The Buckingham Hotel,has_revie

In [8]:
node_creation_cypher_list = []


### Creating A Function for Cypher Query Generation
Based upon the relationships and nodes derived from the text, we instruct GPT engine to generate cypher query
for building a grpah in Neo4j Aura Database


In [9]:
def generate_cypher_for_node_creation(ontology_text):
    cypher_system_prompt = f""" Assistant is an expert in Neo4j Cypher development. Create a cypher query to generate a graph using the data points provided. 
    make sure to only include the cypher query in your response so that I can directly send this cypher query to the Neo4j database API endpoint
    via a POST request. The data is in the format of a comma separated tuple separated by a new line for each tuple.

    """
    
    cypher_user_prompt = f"""Generate a cypher query to create new nodes and their relationships given the data provided. Return only the cypher query. 
    Data is composed of relationships between entities that have been extracted using NER.
    The data is in the format of a comma separated tuple separated by a new line for each tuple.
    
    Example Input: 
    Creek Hotel,is_located_in,Dubai
    Creek Hotel,has_customers,businessmen
    Creek Hotel,has_customers,tourists
    Creek Hotel,has_reviewer,Ryouta Sato
    Creeh Hotel,has_facilities,swimming pool

    Example Output:
    CREATE (ch:Hotel {{name: 'Creek Hotel'}})-[:is_located_in]->(d:Location {{name: 'Dubai'}}),
        (ch)-[:has_customers]->(b:CustomerType {{name: 'businessmen'}}),
        (ch)-[:has_customers]->(t:CustomerType {{name: 'tourists'}}),
        (ch)-[:has_reviewer]->(rs:Reviewer {{name: 'Ryouta Sato'}})
        (ch)-[:has_facilities]->(sp:Facilities {{name: 'swimming pool'}})
        
    strictly stick to the above output format
    
    use distinct variable names for each node and relationship to avoid conflicts

    the data is: {ontology_text}

    """
    
    cypher_query = clean_cypher(chat(cypher_system_prompt, cypher_user_prompt))
    node_creation_cypher_list.append(cypher_query)
    print(cypher_query)

In [10]:
for x in ontology_list:
    generate_cypher_for_node_creation(x)

CREATE (ch:Hotel {name: 'Creek Hotel'})
CREATE (loc1:Location {name: 'Dubai'})
CREATE (loc2:Location {name: 'Bur Dubai'})
CREATE (fac1:Facilities {name: 'sofa'})
CREATE (fac2:Facilities {name: 'TV'})
CREATE (fac3:Facilities {name: 'hot-shower bath'})
CREATE (cust1:CustomerType {name: 'businessmen'})
CREATE (cust2:CustomerType {name: 'tourists'})
CREATE (rev1:Reviewer {name: 'Ryouta Sato'})
CREATE (ch)-[:is_located_in]->(loc1),
       (ch)-[:is_located_in]->(loc2),
       (ch)-[:has_facilities]->(fac1),
       (ch)-[:has_facilities]->(fac2),
       (ch)-[:has_facilities]->(fac3),
       (ch)-[:has_customers]->(cust1),
       (ch)-[:has_customers]->(cust2),
       (ch)-[:has_reviewer]->(rev1);
CREATE (bh:Hotel {name: 'The Buckingham Hotel'})-[:is_located_in]->(n:Location {name: 'New York City'}), 
       (bh)-[:has_facilities]->(f1:Facilities {name: 'sofa'}), 
       (bh)-[:has_facilities]->(f2:Facilities {name: 'TV'}), 
       (bh)-[:has_facilities]->(f3:Facilities {name: 'hot-shower ba

In [11]:
for x in node_creation_cypher_list:
    print(x)

CREATE (ch:Hotel {name: 'Creek Hotel'})
CREATE (loc1:Location {name: 'Dubai'})
CREATE (loc2:Location {name: 'Bur Dubai'})
CREATE (fac1:Facilities {name: 'sofa'})
CREATE (fac2:Facilities {name: 'TV'})
CREATE (fac3:Facilities {name: 'hot-shower bath'})
CREATE (cust1:CustomerType {name: 'businessmen'})
CREATE (cust2:CustomerType {name: 'tourists'})
CREATE (rev1:Reviewer {name: 'Ryouta Sato'})
CREATE (ch)-[:is_located_in]->(loc1),
       (ch)-[:is_located_in]->(loc2),
       (ch)-[:has_facilities]->(fac1),
       (ch)-[:has_facilities]->(fac2),
       (ch)-[:has_facilities]->(fac3),
       (ch)-[:has_customers]->(cust1),
       (ch)-[:has_customers]->(cust2),
       (ch)-[:has_reviewer]->(rev1);
CREATE (bh:Hotel {name: 'The Buckingham Hotel'})-[:is_located_in]->(n:Location {name: 'New York City'}), 
       (bh)-[:has_facilities]->(f1:Facilities {name: 'sofa'}), 
       (bh)-[:has_facilities]->(f2:Facilities {name: 'TV'}), 
       (bh)-[:has_facilities]->(f3:Facilities {name: 'hot-shower ba

In [12]:
%pip install neo4j

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
print(node_creation_cypher_list[0])

CREATE (ch:Hotel {name: 'Creek Hotel'})
CREATE (loc1:Location {name: 'Dubai'})
CREATE (loc2:Location {name: 'Bur Dubai'})
CREATE (fac1:Facilities {name: 'sofa'})
CREATE (fac2:Facilities {name: 'TV'})
CREATE (fac3:Facilities {name: 'hot-shower bath'})
CREATE (cust1:CustomerType {name: 'businessmen'})
CREATE (cust2:CustomerType {name: 'tourists'})
CREATE (rev1:Reviewer {name: 'Ryouta Sato'})
CREATE (ch)-[:is_located_in]->(loc1),
       (ch)-[:is_located_in]->(loc2),
       (ch)-[:has_facilities]->(fac1),
       (ch)-[:has_facilities]->(fac2),
       (ch)-[:has_facilities]->(fac3),
       (ch)-[:has_customers]->(cust1),
       (ch)-[:has_customers]->(cust2),
       (ch)-[:has_reviewer]->(rev1);


### Creating Graph in Neo4j Aura Database


In [14]:
from neo4j import GraphDatabase
import os

uri = os.getenv("NEO4J_URI")

url = uri

neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(url, auth=(neo4j_username, neo4j_password))

with driver.session(database=os.getenv("NEO4J_DATABASE")) as session:
        for cypher_query in node_creation_cypher_list:
            session.run(cypher_query)
            print(f"Executed: {cypher_query}")




Executed: CREATE (ch:Hotel {name: 'Creek Hotel'})
CREATE (loc1:Location {name: 'Dubai'})
CREATE (loc2:Location {name: 'Bur Dubai'})
CREATE (fac1:Facilities {name: 'sofa'})
CREATE (fac2:Facilities {name: 'TV'})
CREATE (fac3:Facilities {name: 'hot-shower bath'})
CREATE (cust1:CustomerType {name: 'businessmen'})
CREATE (cust2:CustomerType {name: 'tourists'})
CREATE (rev1:Reviewer {name: 'Ryouta Sato'})
CREATE (ch)-[:is_located_in]->(loc1),
       (ch)-[:is_located_in]->(loc2),
       (ch)-[:has_facilities]->(fac1),
       (ch)-[:has_facilities]->(fac2),
       (ch)-[:has_facilities]->(fac3),
       (ch)-[:has_customers]->(cust1),
       (ch)-[:has_customers]->(cust2),
       (ch)-[:has_reviewer]->(rev1);
Executed: CREATE (bh:Hotel {name: 'The Buckingham Hotel'})-[:is_located_in]->(n:Location {name: 'New York City'}), 
       (bh)-[:has_facilities]->(f1:Facilities {name: 'sofa'}), 
       (bh)-[:has_facilities]->(f2:Facilities {name: 'TV'}), 
       (bh)-[:has_facilities]->(f3:Facilities {

### Defining function for generation of cypher query to retrieve nodes and context based upon user query

In [15]:
def query_neo4j_graph(user_query):
    query_with_cypher_system_prompt = f"""Assistant is an expert in Neo4j Cypher development. Only return a cypher query based on the user query
    the cypher graph has the following schema:

    Nodes:
    - Hotel
    - Location
    - Facilities
    - CustomerType
    - Reviewer

    Relationships:
    - [Hotel],is_located_in,[Location]
    - [Hotel],has_facilities,[Facilities]
    - [Hotel],has_customers,[CustomerType]
    - [Hotel],has_reviewer,[Reviewer]

    example of a node created through cypher query:
    {node_creation_cypher_list[0]}
    
    Example Input:
    what hotels are reviewed by Ryouta Sato?
    
    Example Output:
    MATCH (h:Hotel)-[:has_reviewer]-(r:Reviewer {{name: 'Ryouta Sato'}})
    RETURN h

    stick strictly to the above output format
    """

    query_with_cypher_user_prompt = f"""Generate a cypher query to answer the user query.
    user_query = {user_query}"""
    
    cypher_query_for_retrieval = clean_cypher(chat(query_with_cypher_system_prompt, query_with_cypher_user_prompt))

    print(cypher_query_for_retrieval)

    return cypher_query_for_retrieval

In [16]:
user_query = "which hotels are visited by businessmen?"
cypher_query_for_retrieval = query_neo4j_graph(user_query)

MATCH (h:Hotel)-[:has_customers]-(c:CustomerType {name: 'businessmen'})
RETURN h


In [17]:
import json
uri = os.getenv("NEO4J_URI")

url = uri

neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(url, auth=(neo4j_username, neo4j_password))
with driver.session(database=os.getenv("NEO4J_DATABASE")) as session:
        # Run the Cypher query
        result = session.run(cypher_query_for_retrieval)
        
        # Extract and print results
        records = [record.data() for record in result]
        print(records)

[{'h': {'name': 'Creek Hotel'}}, {'h': {'name': 'Creek Hotel'}}]


### Sending a final call to GPT engine for summarisation

In [18]:
final_system_prompt = f"""" you are an assistant made to help people. You will be provided with the results of 
a cypher query that returns data from a neo4j database. The results are in the form of list with each object inside
the list containing the final result. Answer the user query in a friendly and an easy to understand manner. 

Note that you will be provided with both the user query that triggered the result and the result itself."""

final_user_prompt = f"""Answer the user query using the results of the cypher query provided below.
user_query = {user_query}
result = {records}"""

print(chat(final_system_prompt, final_user_prompt, temperature=0.7))

Sure! Based on the data, the hotel that’s visited by businessmen is **Creek Hotel**. (It appears more than once in the results, confirming its relevance.)
